In [11]:
import pandas as pd
import numpy as np

In [12]:
# 1. 데이터 로드
life_df = pd.read_csv('./data/Life_Expectancy_Data.csv')
gdp_df = pd.read_csv('./data/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_155.csv', skiprows=4)

In [13]:
# 컬럼명 앞뒤 공백 제거 (매우 중요: 'Life expectancy ' 같은 공백 방지)
life_df.columns = [col.strip() for col in life_df.columns]

In [14]:
# 2. 월드뱅크 GDP 데이터 변환 (가로 -> 세로)
years = [str(y) for y in range(2000, 2016)]
gdp_melted = gdp_df.melt(id_vars=['Country Name'], value_vars=years, 
                         var_name='Year', value_name='Total_GDP')
gdp_melted['Year'] = gdp_melted['Year'].astype(int)

In [15]:
# 3. 국가명 매칭 딕셔너리 (작성하신 내용 수정 보완)
name_map = {
    "Democratic People's Republic of Korea": "Korea, Dem. People's Rep.",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.",
    "Egypt": "Egypt, Arab Rep.",
    "Gambia": "Gambia, The",
    "Iran (Islamic Republic of)": "Iran, Islamic Rep.",
    "Lao People's Democratic Republic": "Lao PDR",
    "Micronesia (Federated States of)": "Micronesia, Fed. Sts.",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Republic of Korea": "Korea, Rep.",
    "Republic of Moldova": "Moldova",
    "Saint Lucia": "St. Lucia",
    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "Slovakia": "Slovak Republic",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "United States of America": "United States",
    "Venezuela (Bolivarian Republic of)": "Venezuela, RB",
    "Viet Nam": "Vietnam",
    "Yemen": "Yemen, Rep.",
    "United Republic of Tanzania": "Tanzania",
    "Bolivia (Plurinational State of)": "Bolivia",
    "Bahamas": "Bahamas, The"
}
life_df['Country'] = life_df['Country'].str.strip().replace(name_map)

In [19]:
# 4. GDP 전처리 (병합 및 단위 변환)
merged = pd.merge(life_df, gdp_melted, left_on=['Country', 'Year'], right_on=['Country Name', 'Year'], how='left')

In [24]:
# 6. [중요] 나눗셈 에러 방지 (Population이 0이거나 NaN인 경우)
# 인구수가 0보다 큰 데이터만 계산하고, 나머지는 NaN 처리
merged['Total_GDP'] = pd.to_numeric(merged['Total_GDP'], errors='coerce')
merged['Population'] = pd.to_numeric(merged['Population'], errors='coerce')
merged['GDP'] = pd.to_numeric(merged['GDP'], errors='coerce')

In [25]:
# 7. 나눗셈 실행 (Population이 NaN이거나 0인 경우 제외)
merged['Calculated_GDP'] = np.where(
    (merged['Population'] > 0) & (merged['Total_GDP'].notnull()), 
    merged['Total_GDP'] / merged['Population'], 
    np.nan
)

In [26]:
# 8. 원래 빈 곳 채우기
merged['GDP'] = merged['GDP'].fillna(merged['Calculated_GDP'])

In [27]:
# 9. 결과 확인 및 저장
final_df = merged.dropna(subset=['GDP'])
final_df.to_csv('Life_Expectancy_Fixed_Final.csv', index=False)

In [31]:
f_df = pd.read_csv('./data/Life_Expectancy_Fixed_Final.csv')

FileNotFoundError: [Errno 2] No such file or directory: './data/Life_Expectancy_Fixed_Final.csv'